In [ ]:
!pip install nltk scikit-learn

import pandas as pd
import numpy as np
import re
import nltk

from google.colab import files

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')
nltk.download('wordnet')


print("📂 Upload your dataset (IMDB Dataset.csv)")
uploaded = files.upload()


filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("\n✅ Dataset Loaded")
print(df.head())

print("\nTotal Samples:", len(df))
print("\nClass Distribution:")
print(df['sentiment'].value_counts())


df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

df['clean_text'] = df['review'].apply(preprocess_text)

print("\n🔍 Before vs After Preprocessing:")
print(df[['review','clean_text']].head())


bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

y = df['sentiment']


X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)



# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)
y_pred_dt = dt.predict(X_test_tfidf)

def evaluate(y_true, y_pred, name):
    print("\n====================")
    print(name)
    print("====================")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1 Score :", f1_score(y_true, y_pred))

evaluate(y_test, y_pred_lr, "Logistic Regression (TF-IDF)")
evaluate(y_test, y_pred_nb, "Naive Bayes (BoW)")
evaluate(y_test, y_pred_dt, "Decision Tree (TF-IDF)")

print("\n✅ DONE")

📂 Upload your dataset (IMDB Dataset.csv)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
